In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.12.0.1', 'jax==0.12.0.1', 'jax[cuda12]==0.12.0.1', 'exact'), 'jaxlib': ('0.12.0.1', 'jaxlib==0.12.0.1', 'jaxlib==0.12.0.1', 'exact'), 'flax': ('0.12.12', 'flax==0.12.12', 'flax==0.12.12', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "8a5255b0fce49fc730baae6ea6ecc601d45bb16b"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/8a5255b0fce49fc730baae6ea6ecc601d45bb16b/d2l"
for name in ('__init__.py', 'jax.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# The Transformer Block

that section built attention, a layer in which each position
reads from all the others. On its own it makes a weak model. Each output of an
attention layer is a weighted average of value vectors, which are linear
projections of the inputs. The softmax weights decide how much of each value
to take, but nothing in the layer computes a nonlinear feature of what a
single position holds. A language model needs that computation at every
position, and it needs it stacked dozens of layers deep. Depth brings a
problem of its own: a signal that passes through 32 layers can shrink,
explode, or lose the differences between tokens before it reaches the
output, and the gradient makes the same trip in reverse.

The *transformer block* answers both needs. Attention moves information
between positions, a position-wise feed-forward network transforms each
position on its own, and both add their outputs to a shared residual stream
that carries the signal through the stack. This section studies the two
design choices that separate the 2017 block from current ones: where the
normalization sits and what form the feed-forward network takes. We measure
the first at initialization, where it decides whether a 32-block stack keeps
its tokens distinct and its attention trainable. We measure the second in a
small language model at matched parameter count. A configurable
`TransformerBlock` class then collects these choices, and the rest of the
chapter builds on it.

In [ ]:
import math
import numpy as np
import jax
from jax import numpy as jnp
from flax import nnx
import optax
from d2l import jax as d2l

## Components of a Transformer Block

A transformer block contains two sublayers connected by a residual stream.
that section introduced the *residual stream* view:
each token carries a running vector $\mathbf{x} \in \mathbb{R}^d$ from layer
to layer. Sublayers never overwrite this vector. They read from it,
compute, and *add* their result back. The two sublayers of the block divide
the work as follows.

- **Attention** is the only place where positions interact. Each token
  queries the others as in that section and adds the
  retrieved mixture to its stream. Contextual information enters through
  this sublayer.
- The **feed-forward network** (FFN) acts on each position separately: the
  same two- or three-matrix MLP is applied token by token, with no
  interaction between positions. It transforms each token's accumulated
  state and contains most of the block's parameters.

Writing $\mathrm{Attn}$ for multi-head self-attention and $\mathrm{Norm}$
for a normalization layer, the modern (*pre-norm*) block computes

$$
\mathbf{h} = \mathbf{x} + \mathrm{Attn}(\mathrm{Norm}_1(\mathbf{x})), \qquad
\mathbf{y} = \mathbf{h} + \mathrm{FFN}(\mathrm{Norm}_2(\mathbf{h})).
$$

Both sublayers preserve the shape $(n, d)$. The block maps a sequence of
$d$-dimensional vectors to a sequence of the same length and width, so
dozens of copies can be stacked. The block also knows nothing about the
order of the tokens. Attention is permutation equivariant
(that section), and no other part of the block
depends on order. The model around the blocks must supply positions, and
the next section shows how.

the figure shows the wiring, in the two normalization
arrangements compared below.

![The transformer block: attention and a position-wise FFN read from the residual stream and add their results back. Left, the 2017 arrangement: normalization sits on the stream itself, after each addition. Right, the modern arrangement: normalization moves onto each branch, and the stream runs uninterrupted from input to output.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-transformers-block-anatomy.svg)

## Normalization Placement

Residual connections and normalization entered the block together in 2017
[@Vaswani.Shazeer.Parmar.ea.2017]. Without some control on scale, the
sum of dozens of sublayer outputs drifts, and deep stacks become hard to
train. The standard normalizer is *layer normalization*
[@Ba.Kiros.Hinton.2016] (that section):
it subtracts the mean of each token's $d$ features, divides by their
standard deviation, and then applies a learned scale and shift per feature.
Unlike batch normalization (that section), it uses no batch
statistics, so it behaves identically in training and inference and does
not depend on the sequence length. The 2017 paper also fixed *where* the
normalization goes, and later work revised that choice.

### Two Arrangements

The original block normalizes the stream *after* each addition
("post-LN"):

$$
\mathbf{h} = \mathrm{Norm}_1(\mathbf{x} + \mathrm{Attn}(\mathbf{x})), \qquad
\mathbf{y} = \mathrm{Norm}_2(\mathbf{h} + \mathrm{FFN}(\mathbf{h})).
$$

The modern block the equation normalizes each branch's *input*
("pre-LN") and leaves the stream alone. In code the two differ by two
lines. The minimal block below takes the arrangement as a flag. Its
attention is the `d2l.MultiHeadAttention` of
that section, and its FFN is the classic two-matrix
MLP with the GELU activation of that section. Later in this section we
upgrade both ingredients and collect every option in one configurable
class; until then this short block keeps the two arrangements side by side
and nothing else in view.

In [ ]:
class MiniBlock(nnx.Module):
    """Attention + MLP on a residual stream; norm placement as a flag."""
    def __init__(self, num_hiddens, pre_norm, *, rngs):
        self.pre_norm = pre_norm
        self.attention = d2l.MultiHeadAttention(num_hiddens, 4, rngs=rngs)
        self.W_1 = d2l.Linear(num_hiddens, 4 * num_hiddens, rngs=rngs)
        self.W_2 = d2l.Linear(4 * num_hiddens, num_hiddens, rngs=rngs)
        self.norm1 = nnx.LayerNorm(num_hiddens, epsilon=1e-5, rngs=rngs)
        self.norm2 = nnx.LayerNorm(num_hiddens, epsilon=1e-5, rngs=rngs)

    def ffn(self, X):  # exact GELU
        return self.W_2(jax.nn.gelu(self.W_1(X), approximate=False))

    def __call__(self, X):
        if self.pre_norm:
            Y = self.norm1(X)
            X = X + self.attention(Y, Y, Y)
            return X + self.ffn(self.norm2(X))
        X = self.norm1(X + self.attention(X, X, X))
        return self.norm2(X + self.ffn(X))

### Signal Propagation at Initialization

Much of the difference between the two arrangements shows before any
training. A freshly initialized network is a fixed random function, so we
can *measure* how it treats signals. We stack 32 blocks, give every weight
matrix the Xavier initialization of the original transformer
(that section), pass a random sequence through the stack, and
record three quantities at every depth: the scale of the residual stream,
how *distinct* the tokens remain from one another, and how much gradient
each block's attention receives from a loss at the top. The numbers change
little from one random draw to the next.

For distinctness we use the cross-token spread
$\|\mathbf{H} - \bar{\mathbf{H}}\| / \|\mathbf{H}\|$, where
$\mathbf{H}$ holds the token vectors of one sequence as rows and
$\bar{\mathbf{H}}$ copies their average to every row. The spread is about
$1$ for unrelated random tokens and $0$ when all tokens have collapsed onto a
single point. The loss takes some care. After a final normalization layer,
the obvious probe $\|\mathbf{H}\|^2$ is constant by construction and has
zero gradient, so we use a generic linear readout
$\langle \mathbf{R}, \mathbf{H} \rangle$ for a fixed random
$\mathbf{R}$ instead.

In [ ]:
def signal_stats(pre_norm, num_blks=32, num_hiddens=256):
    """Token spread, stream RMS, and gradient norms of a fresh stack."""
    rngs = nnx.Rngs(d2l.get_key())
    stack = nnx.Sequential(*[MiniBlock(num_hiddens, pre_norm, rngs=rngs)
                             for _ in range(num_blks)])
    xavier = nnx.initializers.xavier_uniform()
    for _, m in nnx.iter_graph(stack):
        if isinstance(m, nnx.Linear):   # the 2017 initialization
            m.kernel[...] = xavier(d2l.get_key(), m.kernel.shape)
    X = jax.random.normal(d2l.get_key(), (2, 64, num_hiddens))
    R = jax.random.normal(d2l.get_key(), X.shape)
    H, spread, rms = X, [], []
    for blk in stack.layers:
        H = blk(H)
        spread.append(jnp.linalg.norm(H - H.mean(1, keepdims=True))
                      / jnp.linalg.norm(H))
        rms.append(jnp.sqrt((H ** 2).mean()))
    g = nnx.grad(lambda m: (m(X) * R).mean())(stack)  # generic readout
    gnorm = lambda k, n: jnp.linalg.norm(
        g['layers'][k]['attention'][n]['kernel'][...])
    grad_q = [gnorm(k, 'W_q') for k in range(num_blks)]
    proj = {n: float(gnorm(num_blks - 1, n))
            for n in ('W_q', 'W_k', 'W_v', 'W_o')}
    return {'spread': jnp.stack(spread).tolist(),
            'rms': jnp.stack(rms).tolist(),
            'grad_q': jnp.stack(grad_q).tolist(), 'proj': proj}

We run it for both arrangements, print the stream RMS and the token spread
at a few depths, and plot the gradient of each block's query projection:

In [ ]:
stats = {pre: signal_stats(pre) for pre in (False, True)}
for pre, name in ((False, 'post-LN'), (True, 'pre-LN')):
    s = stats[pre]
    print(f'{name:>8}: stream RMS at k=32: {s["rms"][-1]:5.2f}, '
          'token spread at k=8,16,32: '
          + ' '.join(f'{s["spread"][k-1]:.1e}' for k in (8, 16, 32)))
    print('          grad norms at k=32: '
          + '  '.join(f'{n} {g:7.1e}' for n, g in s['proj'].items()))
d2l.plot(list(range(1, 33)), [stats[False]['grad_q'], stats[True]['grad_q']],
         'block index k', 'query projection gradient norm',
         legend=['post-LN', 'pre-LN'], yscale='log')

The two arrangements produce different networks from the same weights.
Post-LN renormalizes the stream after every addition, so its root mean
square (RMS) is pinned at $1$ at every depth. Pre-LN only ever adds, so its
stream grows like the square root of the depth. At initialization the block
outputs are nearly uncorrelated and their variances add, which predicts an
RMS after 32 blocks of $\sqrt{33} \approx 5.7$. The measured value, between
5 and 5.6, falls somewhat short; an exercise asks why. This growth stabilizes the
stack. Each pre-LN branch reads the stream through its own normalization,
so its output has a fixed scale. As the stream grows, the *relative*
contribution of each successive block therefore shrinks.

The post-LN stack has no such damping, and the token spread shows the
consequence. At initialization attention is nearly uniform, so every
attention sublayer pulls each token toward the average token. Post-LN then
renormalizes, which locks in the contraction, and the spread decays
*geometrically*: by block 32 the tokens agree to about one part in ten
thousand. Pre-LN dilutes the same contraction by the growing stream, and its
spread falls slowly, to between 0.3 and 0.4 after 32 blocks. This
contraction is related to the *rank collapse* that
@Dong.Cordonnier.Loukas.2021 prove for stacks of attention layers
without residual connections, where the outputs converge to a single token
with depth.

Collapsed tokens starve attention of gradient. When every token is nearly
the same vector, every query scores every key alike, so the attention
weights, and with them the loss, barely depend on $\mathbf{W}_q$ and
$\mathbf{W}_k$. At block 32 the post-LN query and key projections receive
gradients of order $10^{-9}$, five to six orders of magnitude less than in
the pre-LN stack, while the value and output projections, which act on the
averaged token itself, still receive gradients of several hundredths. These are
measurements at initialization, but they agree with what
@xiong2020layer find in training: post-LN needs a learning-rate
warmup to train well, and pre-LN trains without one. We use pre-norm as the
default and compare both placements in training in that section.

### RMSNorm

Modern models also simplified the normalizer itself. LayerNorm centers each
token vector, divides it by its standard deviation, and then scales and
shifts it. *RMSNorm* [@Zhang.Sennrich.2019] drops the centering and the
shift and keeps only the part that controls the scale:

$$
\mathrm{RMSNorm}(\mathbf{x}) = \frac{\mathbf{x}}{\sqrt{\tfrac{1}{d} \sum_{i=1}^{d} x_i^2 + \epsilon}} \odot \boldsymbol{\gamma}.
$$

It is a one-line function, and both frameworks ship it. We check our
implementation against the built-in and time the two normalizers with
`d2l.Benchmark`, which that section introduces: it calls a
function a few times to warm up, then reports the average time of further
calls, waiting for the device to finish before it stops the clock.

In [ ]:
def rms_norm(X, weight, eps=1e-5):
    return X / jnp.sqrt((X * X).mean(-1, keepdims=True) + eps) * weight

X = jax.random.normal(d2l.get_key(), (64, 512, 1024))
layernorm = nnx.LayerNorm(1024, epsilon=1e-5, rngs=nnx.Rngs(d2l.get_key()))
rmsnorm = nnx.RMSNorm(1024, epsilon=1e-5, rngs=nnx.Rngs(d2l.get_key()))
print(f'ours vs. built-in: max deviation '
      f'{jnp.abs(rms_norm(X, rmsnorm.scale[...]) - rmsnorm(X)).max():.1e}')
for name, f in (('LayerNorm', nnx.jit(layernorm)),
                ('RMSNorm', nnx.jit(rmsnorm))):
    print(d2l.Benchmark(lambda: f(X), warmup=10, repeats=200, desc=name))

Our implementation agrees with the built-in to rounding error. RMSNorm
computes one statistic per token instead of two and has no shift parameter.
How much time this saves depends on the kernel more than on the formula.
Both normalizers read and write the whole tensor, and when that memory
traffic dominates, as it often does on GPUs, the saved arithmetic changes
little. The two timings above therefore vary with the framework and the
device. RMSNorm is preferred because it does less work at equal quality. @Zhang.Sennrich.2019 match LayerNorm's accuracy across
several tasks while reducing the running time of their models by 7% to 64%,
and Llama adopted it [@touvron2023llama].

### Normalizing Queries and Keys

A further normalization sits *inside* attention. Nothing in the block above
controls the size of the attention logits
$\mathbf{q}^\top \mathbf{k} / \sqrt{p}$, where $p$ is the head width. If
training inflates the queries and keys, the softmax saturates, its Jacobian
vanishes (that section), and training can
spike or stall. The instability was first reported in a 22-billion-parameter
vision transformer, whose attention logits grew until the attention weights
were nearly one-hot [@Dehghani.Djolonga.Mustafa.ea.2023], and
@Wortsman.Liu.Xiao.ea.2024 reproduce it in small language models
trained at high learning rates. *QK-norm*
[@Henry.Dachapally.Pawar.ea.2020] applies RMSNorm to the queries and
the keys of each head right before the dot product, so the size of the
logits no longer grows with the query and key weights. It adds two small
norms per layer, and both studies find that it removes the instability.

Recent open models also place norms on the other side of a sublayer. OLMo 2
normalizes what each sublayer *writes* before adding it to the stream
[@OLMo.2025], and Gemma 3 normalizes each branch both before and after
the sublayer [@Gemma.Team.2025]; both also use QK-norm.
the figure compares the four placements. Only post-LN
normalizes the stream itself; the signal experiment above compared it with
pre-LN. QK-norm is not a flag of our block. It is an exercise, and it
enters the block through the `attn_factory` hook introduced below.

![Four placements of normalization around one sublayer, with the residual stream running bottom to top; the FFN sublayer is treated the same way. Post-LN normalizes the stream itself after each addition, pre-LN normalizes what the branch reads, OLMo 2 normalizes what the branch writes, and Gemma 3 normalizes the branch on both sides. Only post-LN touches the stream's identity path.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-transformers-norm-taxonomy.svg)

## The Feed-Forward Network

Attention moves information between positions but computes no nonlinear
feature of what a single position holds. That computation happens in the
other half of the block, a small MLP applied to each position
independently,

$$
\mathrm{FFN}(\mathbf{x}) = \mathbf{W}_2\, \phi(\mathbf{W}_1 \mathbf{x}),
$$

with $\mathbf{W}_1 \in \mathbb{R}^{4d \times d}$ and $\mathbf{W}_2 \in
\mathbb{R}^{d \times 4d}$ in the classic configuration. The expansion by a
factor of 4 dates from the 2017 transformer and is still the common
default. With it, the FFN holds $8d^2$ parameters against the $4d^2$ of
the four attention projections $\mathbf{W}_q, \mathbf{W}_k, \mathbf{W}_v,
\mathbf{W}_o$, so about two thirds of the parameters of a block sit in these
two matrices. @Geva.Schuster.Berant.ea.2021 read them as a memory with
the structure of attention: each row of $\mathbf{W}_1$ acts as a key that
responds to a pattern in the token's vector, and the matching column of
$\mathbf{W}_2$ is the value that the FFN adds to the stream when its key
responds.

Two changes separate the 2017 FFN from current ones. First, the activation
$\phi$ moved from ReLU to the *GELU* $\phi(z) = z\,\Phi(z)$ of
that section [@Hendrycks.Gimpel.2016], where $\Phi$ is the
standard normal cumulative distribution function. GPT used it from the
start [@Radford.Narasimhan.Salimans.ea.2018]. Second,
@Shazeer.2020 revived an older idea, gating, and found that a
*gated linear unit* with the SiLU activation beat both:

$$
\mathrm{SwiGLU}(\mathbf{x}) = \mathbf{W}_2 \big(\mathrm{SiLU}(\mathbf{W}_g \mathbf{x}) \odot \mathbf{W}_1 \mathbf{x}\big), \qquad \mathrm{SiLU}(z) = z \cdot \mathrm{sigmoid}(z).
$$

Instead of passing a linear transform through a fixed nonlinearity, SwiGLU
computes *two* linear views of the input and multiplies them. One view is
squashed into a soft gate. The other passes through unchanged, so wherever
the gate is open it is transmitted at full strength instead of being bent
by the activation. Gates in LSTMs (that section) use the same
multiplicative interaction. A third matrix means more parameters at the
same width, so fair comparisons shrink the hidden width to
$\tfrac{8}{3} d$. The three matrices of the gated FFN then hold the same
$8d^2$ parameters as the classic two, up to rounding. Both variants fit in
one class:

In [ ]:
class FeedForward(nnx.Module):
    """Position-wise FFN: GELU MLP or SwiGLU at matched parameter count."""
    def __init__(self, num_hiddens, act='swiglu', bias=False, *, rngs):
        assert act in ('swiglu', 'gelu'), f'unknown act: {act!r}'
        self.act = act
        if act == 'gelu':
            width = 4 * num_hiddens
        else:  # 'swiglu': three matrices; width 8d/3 matches the MLP budget
            width = round(8 * num_hiddens / 3)
            self.W_g = d2l.Linear(num_hiddens, width, use_bias=bias,
                                  rngs=rngs)
        self.W_1 = d2l.Linear(num_hiddens, width, use_bias=bias, rngs=rngs)
        self.W_2 = d2l.Linear(width, num_hiddens, use_bias=bias, rngs=rngs)

    def __call__(self, X):
        if self.act == 'gelu':  # The tanh approximation, as in GPT-2
            return self.W_2(jax.nn.gelu(self.W_1(X), approximate=True))
        return self.W_2(jax.nn.silu(self.W_g(X)) * self.W_1(X))

Both tabs use the tanh approximation of GELU, the variant GPT-2 was trained
with, which matters when we load its weights in that section. We check
the parameter match with `d2l.num_params` from that section,
which counts the trainable parameters of a model:

In [ ]:
for act in ('gelu', 'swiglu'):
    ffn = FeedForward(256, act, rngs=nnx.Rngs(d2l.get_key()))
    print(f'{act:>7}: {d2l.num_params(ffn)} parameters')

The two variants differ by 256 parameters, 0.05% of the total. Whether
the gate improves the model at equal size is an empirical question, and
the last experiment of this section answers it in a small language model.

## A Configurable Block

We now assemble the block used throughout this chapter. Every design
decision above becomes a constructor argument: `norm` selects LayerNorm or
RMSNorm, `act` selects the FFN, and `pre_norm` selects the arrangement.
Two factory hooks, `attn_factory` and `ffn_factory`, let later sections
swap whole sublayers without touching this class again: a causal attention,
an attention with a smaller key--value cache, or a mixture-of-experts FFN.
The constructor checks its flag strings, so a typo raises an error instead
of silently selecting another architecture. The default configuration
(pre-norm, RMSNorm, SwiGLU) is the one Llama adopted in 2023
[@touvron2023llama], and many open-weights decoders have kept it since.
`TransformerBlock(num_hiddens, num_heads, norm='layer', act='gelu',
pre_norm=False, bias=True)` is a compact post-norm block in the 2017 encoder
arrangement, with one anachronism kept for simplicity: GELU, where the
original used ReLU.

In [ ]:
class TransformerBlock(nnx.Module):
    """Configurable transformer block: attention + FFN on a residual
    stream."""
    def __init__(self, num_hiddens, num_heads, *, dropout=0, norm='rms',
                 act='swiglu', pre_norm=True, bias=False, attn_factory=None,
                 ffn_factory=None, rngs):
        assert norm in ('rms', 'layer'), f'unknown norm: {norm!r}'
        assert num_hiddens % num_heads == 0
        self.pre_norm = pre_norm
        make_norm = nnx.RMSNorm if norm == 'rms' else nnx.LayerNorm
        self.norm1 = make_norm(num_hiddens, epsilon=1e-5, rngs=rngs)
        self.norm2 = make_norm(num_hiddens, epsilon=1e-5, rngs=rngs)
        self.attention = (d2l.MultiHeadAttention(num_hiddens, num_heads,
                                                 dropout, bias=bias,
                                                 rngs=rngs)
                          if attn_factory is None else attn_factory(rngs))
        self.ffn = (FeedForward(num_hiddens, act, bias=bias, rngs=rngs)
                    if ffn_factory is None else ffn_factory(rngs))
        self.dropout = nnx.Dropout(dropout, rngs=rngs)

    def __call__(self, X, mask=None, **attn_args):
        # attn_args go on to the attention, e.g. the RoPE tables of d2l.GPT
        if self.pre_norm:
            Y = self.norm1(X)
            X = X + self.dropout(self.attention(Y, Y, Y, mask, **attn_args))
            return X + self.dropout(self.ffn(self.norm2(X)))
        X = self.norm1(X + self.dropout(self.attention(X, X, X, mask,
                                                       **attn_args)))
        return self.norm2(X + self.dropout(self.ffn(X)))

### Shapes and Parameters

Whatever the flags, the block preserves the shape of its input. Its
parameter count confirms the accounting above: $4d^2$ in attention, $8d^2$
in the FFN however the activation divides it, and a few normalization
weights. That makes about $12d^2$ per block, two thirds of it in the FFN.

In [ ]:
X = jnp.ones((2, 10, 256))
blk = TransformerBlock(256, num_heads=8, rngs=nnx.Rngs(d2l.get_key()))
blk2017 = TransformerBlock(256, num_heads=8, norm='layer', act='gelu',
                           pre_norm=False, bias=True,
                           rngs=nnx.Rngs(d2l.get_key()))
d2l.check_shape(blk(X), X.shape)
d2l.check_shape(blk2017(X), X.shape)
attn, ffn, total = (d2l.num_params(m) for m in (blk.attention, blk.ffn, blk))
print(f'attention {attn}, ffn {ffn}, total {total} = '
      f'{total / 256 ** 2:.2f} d^2')

At $d = 256$ the block holds $12.01 d^2$ parameters. The same count scales
to GPT-2 small, whose blocks have width $d = 768$: $12 d^2 \approx 7.1$
million parameters per block, and about 85 million in its 12 blocks. Its
token and position embeddings add another 39 million, which brings the
model to its 124 million parameters.

One block is not yet a language model, but a few lines turn a stack of
blocks into one we can train. A boolean mask makes the block *causal*. The
mask `d2l.causal_mask(T)` (that section) lets
the query at position $t$ see the keys at positions $0, \ldots, t$, so
every token attends only backward. A dozen lines wrap an embedding, a stack
of blocks, a final RMSNorm, and a tied output head
(that section) around the mask:

In [ ]:
class CharLM(nnx.Module):
    """Minimal byte-level LM: causal masking through a boolean mask."""
    def __init__(self, vocab_size, num_hiddens=128, num_heads=4, num_blks=2,
                 act='swiglu', max_len=128, *, rngs):
        init = nnx.initializers.normal(0.02)
        self.token_emb = nnx.Embed(vocab_size, num_hiddens,
                                   embedding_init=init, rngs=rngs)
        self.pos_emb = nnx.Embed(max_len, num_hiddens,
                                 embedding_init=init, rngs=rngs)
        self.blks = nnx.List([
            TransformerBlock(num_hiddens, num_heads, act=act, rngs=rngs)
            for _ in range(num_blks)])
        self.norm = nnx.RMSNorm(num_hiddens, epsilon=1e-5, rngs=rngs)

    def __call__(self, X):
        T = X.shape[1]
        H = self.token_emb(X) + self.pos_emb(jnp.arange(T))
        causal = d2l.causal_mask(T)
        for blk in self.blks:
            H = blk(H, causal)
        return self.token_emb.attend(self.norm(H))

### Comparing GELU and SwiGLU

SwiGLU keeps its parameter count equal to the GELU network's by
shrinking the hidden width, so a comparison of the two asks whether the
gate makes a better model at the same size. We train `CharLM` twice on `d2l.WikiText103`, the bytes of the Wikipedia
articles introduced in that section. Both runs use the same
model size, optimizer, learning rate, and budget of 600 steps. Only the
`act` flag differs, and the parameter counts match to a tenth of a percent. Each line
reports the validation loss after 200, 400, and 600 steps, measured on the
held-out articles with `d2l.lm_perplexity` (that section), whose
logarithm is the mean loss in nats per byte.

Two shorthands keep the experiment the same in both frameworks: `charlm`
builds a `CharLM` with fresh random weights, and `adamw` returns AdamW
without weight decay.

In [ ]:
def charlm(*args, **kwargs):  # Fresh random weights for every model
    return CharLM(*args, **kwargs, rngs=nnx.Rngs(d2l.get_key()))

def adamw(model, lr):
    """Plain AdamW without weight decay."""
    return nnx.Optimizer(model, optax.adamw(lr, weight_decay=0.0),
                         wrt=nnx.Param)

In [ ]:
data = d2l.WikiText103(batch_size=64, num_steps=128)
for act in ('gelu', 'swiglu'):
    model = charlm(len(data.vocab), act=act)
    optimizer = adamw(model, lr=1e-3)
    val = []
    for _ in range(3):  # Three rounds of 200 steps
        d2l.train_steps(model, data, optimizer, 200)
        val.append(math.log(d2l.lm_perplexity(model, data.val_dataloader())))
    print(f'{act:>7}: validation loss at step 200/400/600: '
          + '/'.join(f'{v:.2f}' for v in val))

The gated FFN ends a few hundredths of a nat ahead, and it stays ahead when
we repeat the comparison from fresh random initializations. The
direction agrees with the comparison of @Shazeer.2020 at the scale
of T5. There, at matched parameters and computation, SwiGLU lowered the
held-out log-perplexity by about 0.04 nats against GELU. It also agrees
with the choice of the major open model families since Llama. The
improvement at equal size is modest, and a modern block combines several
improvements of this kind.

We can now build a transformer block in any of the configurations used in
practice and predict how it treats signals at initialization. `CharLM` is
still a toy: two blocks, learned absolute positions, and 600 steps of
training. The next section builds a GPT from the same blocks, with rotary
positions, trains it as a language model, and tests the placement of
normalization in training.

## Summary

Attention moves information between positions but computes no nonlinear
feature of any one position. A transformer block therefore pairs it with a
position-wise FFN, which holds about two thirds of the parameters of the
block, and both add their outputs to a residual stream. In the original
post-LN arrangement, normalization pins the scale of the stream. At
initialization a 32-block stack then collapses its tokens geometrically,
and the query and key projections of the top block receive gradients five
to six orders of magnitude smaller than in the pre-norm stack. Pre-norm
normalizes what each branch reads, lets the stream grow like the square
root of the depth, and keeps every block trainable. RMSNorm keeps only the
scale statistic of LayerNorm, and QK-norm bounds the attention logits.
SwiGLU replaces the fixed activation with a learned gate; at matched
parameter count it ends a few hundredths of a nat ahead of GELU in our
byte-level model. The `TransformerBlock` class exposes these choices as
options, with replaceable attention and FFN modules.

## Exercises

1. **Scale invariance.** Let RMSNorm act as in the equation with
   $\boldsymbol{\gamma} = \mathbf{1}$ and $\epsilon = 0$, and write
   $r(\mathbf{x}) = (\tfrac{1}{d} \sum_i x_i^2)^{1/2}$ for the root mean
   square of $\mathbf{x} \in \mathbb{R}^d$.
    1. Show that $\mathrm{RMSNorm}(\alpha \mathbf{x}) =
       \mathrm{RMSNorm}(\mathbf{x})$ for every $\alpha > 0$, and that
       LayerNorm with unit gain and zero shift is also unchanged when the
       same constant is added to every coordinate. Describe the set of
       inputs that each normalizer maps to a given output.
    1. Show that the Jacobian of RMSNorm at $\mathbf{x}$ is
       $r(\mathbf{x})^{-1} (\mathbf{I} - \hat{\mathbf{x}}
       \hat{\mathbf{x}}^\top)$ with $\hat{\mathbf{x}} = \mathbf{x} /
       \|\mathbf{x}\|$. Conclude that the gradient
       $\nabla_{\mathbf{x}} \ell$ is orthogonal to $\mathbf{x}$ for every
       loss $\ell$ that depends on $\mathbf{x}$ only through
       $\mathrm{RMSNorm}(\mathbf{x})$.

1. **Unrolled residual stream.** Number the $2N$ sublayers of an
   $N$-block stack by $j = 1, \ldots, 2N$, write $F_j$ for the $j$-th
   sublayer (attention or FFN) and $\mathrm{Norm}_j$ for its
   normalization, and let $\mathbf{x}_j$ be the residual stream after
   sublayer $j$, so that $\mathbf{x}_0$ is the input.
    1. For the pre-LN arrangement the equation, write
       $\mathbf{x}_{2N}$ as $\mathbf{x}_0$ plus a sum of $2N$ branch
       outputs. Write the Jacobian $\partial \mathbf{x}_{2N} / \partial
       \mathbf{x}_0$ as a product over sublayers, expand the product, and
       identify the term that passes a gradient from the output to the
       input unchanged.
    1. For the post-LN arrangement the equation, the stream is
       $\mathbf{x}_j = \mathrm{Norm}_j(\mathbf{u}_j)$ with $\mathbf{u}_j =
       \mathbf{x}_{j-1} + F_j(\mathbf{x}_{j-1})$. Model every normalization
       as RMSNorm with unit gain and $\epsilon = 0$. Assume that the input
       $\mathbf{x}_0$ has unit root mean square and that each branch output
       is uncorrelated with the stream it is added to. The product of the
       normalization Jacobians alone takes the place of the identity term
       of the pre-LN expansion. Show that this product scales a vector
       orthogonal to every $\mathbf{u}_j$ by $\prod_j r(\mathbf{u}_j)^{-1}$,
       and write this factor as a function of the mean squares
       $c_1, \ldots, c_{2N}$ of the branch outputs. Evaluate it for a
       common mean square $c_j = c$, and compare its dependence on $N$
       with that of the identity term of the pre-LN expansion.
    1. `CharLM` applies a final RMSNorm, `norm`, before its tied output
       head. Describe what the head would receive without it as a pre-LN
       stack grows deeper, and explain why a post-LN stack needs no final
       normalization.

1. [code] **QK-norm as a factory.**
   QK-norm (that section) applies RMSNorm over the head dimension
   $p = d/h$ to every query and key just before the scores are computed.
    1. Write `QKNormAttention`, a subclass of `d2l.MultiHeadAttention`
       whose `forward` (`__call__` in JAX) passes the per-head queries and
       keys produced by `split_heads` through their own RMSNorm each
       before the dot product. Build a `TransformerBlock` with it through
       `attn_factory` and check the output shape.
    1. Show that with unit gains every scaled logit
       $\hat{\mathbf{q}}^\top \hat{\mathbf{k}} / \sqrt{p}$ lies in
       $[-\sqrt{p}, \sqrt{p}]$, and generalize the bound to learned gains
       $\boldsymbol{\gamma}_q$ and $\boldsymbol{\gamma}_k$. Compute the
       largest possible ratio between two attention weights of one query
       for $p = 64$ and unit gains.
    1. Training can inflate the query and key weights. Mimic this at
       initialization: multiply the weights of `W_q` and `W_k` of a
       freshly initialized layer with $d = 256$ and $h = 4$ by
       $\lambda \in \{1, 2, 4, 8\}$, and feed it 64 standard normal
       tokens. Measure the standard deviation of the logits and the mean
       entropy of the attention weights, with and without QK-norm. Compare
       the entropies with $\log 64$, and explain the shape of each curve.

1. **FFN width rounding.** `FeedForward` sets the SwiGLU width to
   $\mathrm{round}(8d/3)$.
    1. Write the exact parameter count of the default
       `TransformerBlock(d, h)` as a function of $d$, and bound the
       relative deviation of its FFN from $8d^2$.
    1. The released 7B configuration of Llama [@touvron2023llama] has
       $d = 4096$ and SwiGLU width $11008$, the smallest multiple of 256
       that is at least $\lfloor 8d/3 \rfloor$. Compute the relative excess
       of its FFN parameters over $8d^2$, and the excess that the same
       rule gives at $d = 256$ and at $d = 128$, the default `num_hiddens`
       of `CharLM`. Explain what the comparison implies for
       parameter-matched experiments with small models.

1. [code] **Causal masks.** `CharLM` passes the boolean mask
   `causal = d2l.causal_mask(T)` to every block, so that the query at
   position $t = 0, \ldots, T - 1$ attends to the keys at positions
   $0, \ldots, t$.
    1. Show by induction over the blocks that the logits of `CharLM` at
       position $t$ depend only on the tokens at positions $0, \ldots, t$.
       Which step of the induction fails if only the first block receives
       `causal` and the other blocks receive no mask?
    1. Give `CharLM` an argument `shift` and build the mask as
       `torch.ones(T, T, dtype=torch.bool).tril(shift)` (PyTorch) or
       `jnp.tril(jnp.ones((T, T), bool), shift)` (JAX), so that the query at
       position $t$ sees the keys at positions $0, \ldots, t + s$, where
       $s$ is the value of `shift`.
       For `shift=-1`, the query at position 0 sees no key, and
       `d2l.masked_softmax` gives it zero weights. For `shift=1` and for
       `shift=-1`, predict the positions whose logits depend on the token
       at position $j$ in a stack of $N$ blocks. Which of the two settings
       can be trained as a next-token predictor?
    1. Check your predictions on freshly initialized models with
       `num_blks` $\in \{1, 2, 4\}$: change the token at position 16 of a
       sequence of 32 random tokens, and find the positions whose logits
       change.

1. [code] **Stream variance budget.** ● In the pre-LN arrangement every
   branch reads a normalized input. If each branch output is uncorrelated
   with the stream it is added to, the mean square of the stream after $k$
   blocks is
   $$1 + \sum_{i=1}^{k} \big(c^\textrm{attn}_i + c^\textrm{ffn}_i\big),$$
   where $c^\textrm{attn}_i$ and $c^\textrm{ffn}_i$ are the mean squares of
   the attention and FFN outputs of block $i$. Use `MiniBlock` with the
   Xavier initialization of `signal_stats` at $d = 256$.
    1. Predict $c^\textrm{ffn}_1$. Compute the variance of the hidden
       pre-activations of the FFN, and evaluate the second moment of GELU
       at that variance numerically.
    1. Predict $c^\textrm{attn}_1$. Treat the $n = 64$ input tokens as
       independent, and treat the attention logits of one query as
       independent Gaussians with the variance that the initialization
       implies.
    1. Record the mean square of each branch output in `signal_stats` and
       plot $c^\textrm{attn}_k$ and $c^\textrm{ffn}_k$ against $k$.
       Compare block 1 with your predictions, and explain the trend of
       $c^\textrm{attn}_k$ using the token spread.
    1. Fit a power law to the stream RMS, once over blocks 1 to 32 and once
       over blocks 8 to 32. Compare both exponents with $1/2$, explain the
       comparison with the variance budget, and state the regime in which
       the square-root growth stated in that section
       holds.

1. [code] **Rate of token collapse.** ● Write the tokens of one sequence
   as $\mathbf{x}_i = \boldsymbol{\mu} + \boldsymbol{\delta}_i$ for
   $i = 1, \ldots, n$, where $\boldsymbol{\mu}$ is the mean token and
   $\sum_i \boldsymbol{\delta}_i = \mathbf{0}$. With the ratio of common
   to deviation energy
   $$\rho = \frac{n \|\boldsymbol{\mu}\|^2}{\sum_i \|\boldsymbol{\delta}_i\|^2},$$
   the token spread of `signal_stats` equals $(1 + \rho)^{-1/2}$. At
   initialization the attention weights are nearly uniform, so a post-LN
   attention sublayer adds approximately the same vector
   $\mathbf{W}_o \mathbf{W}_v \boldsymbol{\mu}$ to every token, where
   $\mathbf{W}_v, \mathbf{W}_o \in \mathbb{R}^{d \times d}$ are the value
   and output projections.
    1. Let the entries of $\mathbf{W}_v$ and $\mathbf{W}_o$ be independent
       with mean zero and variance $\alpha^2 / d$, so that $\alpha = 1$ is
       Xavier initialization of a square matrix. Treat the normalization as
       a rescaling common to all tokens. Show that one attention sublayer
       and its normalization multiply $\rho$ by about $1 + \alpha^4$, and
       derive the spread after $k$ blocks for an input of $n$ independent
       random tokens, ignoring the FFN sublayers.
    1. Rerun `signal_stats` for post-LN with all Xavier weights multiplied
       by $\alpha \in \{1, 0.7, 0.5\}$, once with the full `MiniBlock` and
       once with its FFN sublayer removed. Compare both with your formula
       at blocks 8, 16, and 32, and explain the remaining gap of the full
       block.
    1. GPT-1 stacked 12 post-LN blocks of width $d = 768$ and drew every
       weight from $\mathcal{N}(0, 0.02^2)$
       [@Radford.Narasimhan.Salimans.ea.2018]. Compute $\alpha$ for its
       attention projections, and predict the spread after 12 blocks for
       $n = 64$, both for this initialization and for Xavier
       initialization at the same width. Check both predictions with
       `signal_stats`.

1. [extended] **Learning-rate sensitivity.** ●
   @Wortsman.Liu.Xiao.ea.2024 reproduce the growth of attention
   logits reported for large training runs (that section) in
   small models trained at high learning rates, and find that QK-norm
   removes it. They summarize the final loss $\ell(\eta)$ as a function of
   the learning rate $\eta$ by the *learning-rate sensitivity*
   $$\mathbb{E}_{\eta}\big[\min(\ell(\eta), \ell_0) - \ell^*\big],$$
   where $\ell_0$ is the loss at initialization, $\ell^*$ is the best final
   loss over the learning rates tried, and the average runs over learning
   rates spanning three orders of magnitude.
    1. Give `CharLM` an option that builds its blocks with
       `QKNormAttention` through `attn_factory`. With `num_hiddens=256` and
       `num_blks=6`, train it with and without QK-norm for 1000 steps at
       each learning rate in $\{10^{-4}, 10^{-3.5}, \ldots, 10^{-1}\}$,
       otherwise under the protocol of that section.
    1. Every 100 steps, record the training loss and the largest absolute
       attention logit on a fixed batch. Plot the final loss against
       $\eta$ for both models, and compute the learning-rate sensitivity
       of each curve.
    1. For every learning rate at which the model without QK-norm ends
       worse than the model with it, decide from the recorded attention
       logits whether attention-logit growth explains the gap. If it does
       not, record the largest absolute output logit of both models at the
       same steps, and decide whether its growth explains the gap.

[Discussions](https://d2l.discourse.group/t/36774)